# 7. Regression

Predict the exact NewsGuard score 0–100 (ridge; ModernBERT with squared error on score/100). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets (for Mistral-24B, the layer too: 27 or 40). ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/regression/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "regression"
MODELS = "TF-IDF,Qwen3-0.6B,Qwen3-8B,Mistral-24B,ModernBERT-frozen"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "TF-IDF,Qwen3-0.6B,Qwen3-8B,Mistral-24B,ModernBERT-frozen"
N_JOBS = 8


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

2026-11-13 16:23:01  regression, TF-IDF, fold 1: already done


2026-11-13 16:23:01  regression, TF-IDF, fold 2: already done


2026-11-13 16:23:01  regression, TF-IDF, fold 3: already done


2026-11-13 16:23:01  regression, TF-IDF, fold 4: already done


2026-11-13 16:23:01  regression, TF-IDF, fold 5: already done


2026-11-13 16:23:05  regression, Qwen3-0.6B, fold 1: penalty 10 (on the grid edge), 4 s


... (26 outputs left out)

2026-11-13 16:25:18  regression, ModernBERT-frozen, fold 2: penalty 1000 (inside the grid), 2 s


2026-11-13 16:25:19  regression, ModernBERT-frozen, fold 3: penalty 10 (on the grid edge), 2 s


2026-11-13 16:25:21  regression, ModernBERT-frozen, fold 4: penalty 100 (inside the grid), 2 s


2026-11-13 16:25:23  regression, ModernBERT-frozen, fold 5: penalty 1000 (inside the grid), 2 s


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

TF-IDF chosen penalty per fold: {1: 0.3, 2: 0.3, 3: 0.01, 4: 0.03, 5: 0.01}
Qwen3-0.6B chosen penalty per fold: {1: 10, 2: 10, 3: 10, 4: 100, 5: 10}
Qwen3-8B chosen penalty per fold: {1: 100, 2: 10000, 3: 100, 4: 100, 5: 1000}
Mistral-24B chosen penalty per fold: {1: 100000, 2: 100000, 3: 100000, 4: 10000, 5: 1000}
ModernBERT-frozen chosen penalty per fold: {1: 10, 2: 1000, 3: 10, 4: 100, 5: 1000}


In [6]:
p = pd.concat([pd.read_csv(f) for f in sorted((C.PREDICTIONS / TASK / "Mistral-24B").glob("penalty_fold*.csv"))])
if "layer" in p:
    print("Mistral-24B, layer chosen on the inner folds:", p[p.chosen].set_index("fold").layer.to_dict())
    display(p.pivot_table(index=["layer", "penalty"], columns="fold", values="inner score").round(3))

Mistral-24B, layer chosen on the inner folds: {1: 27, 2: 27, 3: 27, 4: 27, 5: 27}


fold               1      2      3      4      5
layer penalty                                   
27    10       0.668  0.609  0.636  0.672  0.717
      100      0.681  0.626  0.649  0.686  0.720
      1000     0.715  0.653  0.674  0.692  0.751
      10000    0.727  0.665  0.694  0.707  0.746
      100000   0.744  0.688  0.698  0.702  0.742
      1000000  0.714  0.677  0.647  0.664  0.717
40    10       0.689  0.634  0.645  0.650  0.713
      100      0.693  0.637  0.650  0.671  0.712
      1000     0.690  0.641  0.656  0.697  0.714
      10000    0.697  0.661  0.677  0.692  0.724
      100000   0.706  0.663  0.665  0.681  0.712
      1000000  0.659  0.589  0.552  0.590  0.629